In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)

import numpy as np
import category_encoders as ce
import re

from datetime import timedelta
from dateutil.relativedelta import relativedelta

In [2]:
def parse_date(date_str):
    date = pd.to_datetime(date_str, format="%Y%m%d")
    return date.strftime("%Y-%m-%d") if not pd.isna(date) else ''

In [3]:
pickle_path = '/home/andreea.oprescu/projects/paper_rascal/data_wrangling/data'

In [4]:
hospital_type = {
    "Hospital San Rafael":"Hospital concertado",
    "Hospital General Santa María del Puerto":"Hospital concertado",
    "Hospital Virgen Bella":"Hospital concertado",
    "Hospital Virgen de Las Montañas":"Hospital concertado",
    "H Costa del Sol":"Hospital de especialidades",
    "H. Alto Guadalquivir":"Hospital comarcal",
    "H. de Antequera":"Hospital comarcal",
    "H. de Baza":"Hospital comarcal",
    "H. de La Axarquía":"Hospital comarcal",
    "H. de La Línea de la Concepción":"Hospital comarcal",
    "H. de la Serranía":"Hospital comarcal",
    "H. de Poniente":"Hospital comarcal",
    "H. de Riotinto":"Hospital comarcal",
    "H. Infanta Elena":"Hospital comarcal",
    "H. Infanta Margarita":"Hospital comarcal",
    "H. La Inmaculada":"Hospital comarcal",
    "H. La Merced":"Hospital comarcal",
    "H. Punta de Europa":"Hospital comarcal",
    "H. San Agustín":"Hospital comarcal",
    "H. San Juan de Dios del Aljarafe":"Hospital comarcal",
    "H. San Juan de la Cruz":"Hospital comarcal",
    "H. Santa Ana":"Hospital comarcal",
    "H.U. de Jaén":"Hospital de especialidades",
    "H.U. de Jerez de la Frontera":"Hospital de especialidades",
    "H.U. de Puerto Real":"Hospital de especialidades",
    "H.U. Juan Ramón Jiménez":"Hospital de especialidades",
    "H.U. Puerta del Mar":"Hospital de especialidades",
    "H.U. Regional de Málaga":"Hospital regional",
    "H.U. Reina Sofía":"Hospital regional",
    "H.U. San Cecilio":"Hospital de especialidades",
    "H.U. Torrecárdenas":"Hospital de especialidades",
    "H.U. Virgen de la Victoria":"Hospital de especialidades",
    "H.U. Virgen de las Nieves":"Hospital regional",
    "H.U. Virgen de Valme":"Hospital de especialidades",
    "H.U. Virgen del Rocío":"Hospital regional",
    "H.U. Virgen Macarena":"Hospital regional",
    "H. Valle de los Pedroches":"Hospital comarcal" 
}

## Load data from pickle (already ingested since it takes too long to get directly from df)

In [5]:
socio_demo = pd.read_pickle(pickle_path+'/socio_demo.pickle')
ingreso_hosp = pd.read_pickle(pickle_path+'/ingreso_hosp.pickle')
pmhx = pd.read_pickle(pickle_path+'/pmhx.pickle')
lab = pd.read_pickle(pickle_path+'/lab.pickle')
vacunas = pd.read_pickle(pickle_path+'/vacunas.pickle')
periodos = pd.read_csv(pickle_path+'/05_Periodos_pandemicos.csv', sep=';')

In [6]:
lab["DESC_CLC"].unique().tolist()

['Creatinina',
 'Glucosa',
 'Aspartato transaminasa',
 'Alanina transaminasa',
 'Potasio',
 'Sodio',
 'Hematocrito',
 'Hemoglobina',
 'Linfocitos (porcentaje)',
 'Linfocitos (recuento)',
 'Hemoglobina corpuscular media',
 'Volumen corpuscular medio',
 'Neutrófilos (porcentaje)',
 'Neutrófilos (recuento)',
 'Plaquetas (recuento)',
 'Hematíes (recuento)',
 'Leucocitos (recuento)',
 'Urea',
 'Lactato deshidrogenasa',
 'Proteína C reactiva',
 'Tiempo de protrombina normalizado (INR)',
 'Dímero-D',
 'Tiempo de protrombina',
 'Sodio (sangre total)',
 'Potasio (sangre total)',
 'Urea (sangre total)',
 'Creatinina (sangre total)']

In [7]:
ingreso_hosp.rename(columns={'IND_INGRESO_UCI': 'uci', 'NUM_DIAS_UCI': 'u_stay'}, inplace=True)
ingreso_hosp['centro'] = ingreso_hosp['centro'].str.strip() # remove whitespaces from the beginning and end of the strings
ingreso_hosp.drop(columns=['COD_HOSPITAL_NORMALIZADO'], inplace=True)

In [8]:
ingreso_hosp["centro"].value_counts()

centro
H.U. Virgen del Rocío                      7491
H.U. Virgen de las Nieves                  5024
H.U. Virgen de la Victoria                 4029
H.U. Reina Sofía                           3515
H.U. San Cecilio                           3247
H.U. Virgen Macarena                       3047
H.U. Regional de Málaga                    2291
H.U. Torrecárdenas                         2273
H.U. Virgen de Valme                       2272
H.U. de Jaén                               2047
H.U. Juan Ramón Jiménez                    1876
H.U. de Jerez de la Frontera               1734
H. Infanta Margarita                       1716
H. Costa del Sol                           1715
H.U. Puerta del Mar                        1623
H. de Poniente                             1451
H. Infanta Elena                           1306
H. San Juan de la Cruz                     1273
H.U. de Puerto Real                        1155
H. La Merced                               1098
H. San Juan de Dios del Aljarafe 

In [9]:
# Marcamos con 0 si es múltiple (duplicado) y con 1 si es estancia única
ingreso_hosp['es_estancia_unica'] = (~ingreso_hosp.duplicated(subset="NUHSA_ENCRIPTADO", keep=False)).astype(int)

In [10]:
patient = pd.merge(socio_demo, ingreso_hosp, on='NUHSA_ENCRIPTADO', how='inner') 

### Readmissions: merge into one stay multiple stays of patients who have been discharge and admitted again in the same hospital in less than a day

In [11]:
# 1. Ordenar por paciente, centro y fecha (el orden es crítico para el shift)
patient = patient.sort_values(["NUHSA_ENCRIPTADO", "centro", "admission_datetime"])

# 2. Calcular la fecha de alta anterior dentro del mismo hospital
patient["prev_discharge"] = patient.groupby(["NUHSA_ENCRIPTADO", "centro"])["discharge_datetime"].shift()

# 3. Calcular la diferencia en días
patient["diff"] = (patient["admission_datetime"] - patient["prev_discharge"]).dt.days

# 4. Crear los grupos de fusión:
# Se crea un grupo nuevo si:
# - La diferencia es mayor a 1 día (o es el primer ingreso: NaT)
# - Cambia el paciente
# - Cambia el hospital
patient["group"] = (
    (patient["diff"] > 1) | 
    (patient["diff"].isna()) |
    (patient["NUHSA_ENCRIPTADO"] != patient["NUHSA_ENCRIPTADO"].shift()) |
    (patient["centro"] != patient["centro"].shift())
).cumsum()

# 5. Fusionar registros agrupados
patient = (
    patient.groupby("group", as_index=False)
    .agg({
        **{c: "first" for c in patient.columns if c not in ["admission_datetime", "discharge_datetime", "group", "prev_discharge", "diff"]},
        "admission_datetime": "min",
        "discharge_datetime": "max"
    })
)
# Ahora que hemos fusionado filas, recalculamos quién es realmente "único"
patient['es_estancia_unica'] = (~patient.duplicated(subset="NUHSA_ENCRIPTADO", keep=False)).astype(int)
# 6. Eliminar columnas auxiliares
patient = patient.drop(columns=["prev_discharge", "diff", "group"], errors="ignore")

In [12]:
print("Number of patients after merging cases that get discharge and admitted again in the same hospital:")
print(len(patient))

Number of patients after merging cases that get discharge and admitted again in the same hospital:
58661


In [13]:
# Calculamos los días de hospitalización
patient['hospitalized_days'] = (patient['discharge_datetime'] - patient['admission_datetime']).dt.days

# Ajuste clínico:
# En gestión hospitalaria, si un paciente ingresa y se va el mismo día, 
# la resta da 0, pero suele contarse como 1 día de ocupación de cama.
patient['hospitalized_days'] = patient['hospitalized_days'].apply(lambda x: 1 if x == 0 else x)
print(patient['hospitalized_days'].describe())

count    58661.000000
mean        11.241847
std         12.884025
min          1.000000
25%          5.000000
50%          8.000000
75%         13.000000
max        404.000000
Name: hospitalized_days, dtype: float64


In [14]:
# Convertimos a string, reemplazamos '-1' por nulo y luego a datetime
patient['death_datetime'] = pd.to_datetime(
    patient['COD_FEC_FALLECIMIENTO'].astype(str).replace('-1', np.nan), 
    format='%Y%m%d', 
    errors='coerce'
)
# Creamos la columna target (objetivo)
patient['hospital_outcome'] = np.where(
    (patient['death_datetime'].notna()) & 
    (patient['death_datetime'] >= patient['admission_datetime']) & 
    ((patient['death_datetime'] - patient['admission_datetime']).dt.days <= 30),
    1,
    0
)

In [15]:
print(patient['hospital_outcome'].value_counts())

hospital_outcome
0    48219
1    10442
Name: count, dtype: int64


In [16]:
patient.drop(columns=['COD_FEC_FALLECIMIENTO','DESC_SEXO'], inplace=True)

### Exclusion criteria applied: hospitals with less than 100 patients

In [17]:
patient = patient[patient.groupby('centro')['centro'].transform('count') >= 100].copy()
print("Number of patients after excluding hospitals with less than 100 patients:")
print(len(patient))

Number of patients after excluding hospitals with less than 100 patients:
58622


In [18]:
print("Get age")
patient['age'] = (patient['admission_datetime'] - patient['COD_FEC_NACIMIENTO']).dt.days // 365

Get age


### Exclusion criteria applied: 
#### A) hospitals with more than 50% missing lab data 


In [19]:
# 1. Identificamos quiénes SÍ están en lab
esta_en_lab = patient['NUHSA_ENCRIPTADO'].isin(lab['NUHSA_ENCRIPTADO'])

# 2. Calculamos el porcentaje de borrados por centro (antes de borrar nada)
reporte_centros = (1 - esta_en_lab.groupby(patient['centro']).mean()) * 100
reporte_centros = reporte_centros.reset_index(name='Porcentaje_Borrado')

# 3. Identificamos centros con más del 50% de missing
centros_a_eliminar = reporte_centros[reporte_centros['Porcentaje_Borrado'] > 50]['centro'].tolist()

print(f"Centros que se van fuera por falta de datos (>50%): {centros_a_eliminar}")

Centros que se van fuera por falta de datos (>50%): ['H. Alto Guadalquivir', 'H. Costa del Sol', 'H. San Juan de Dios del Aljarafe', 'H. de Poniente']


In [20]:
display(reporte_centros)

,centro,Porcentaje_Borrado
0,H. Alto Guadalquivir,66.003976
1,H. Costa del Sol,96.781744
2,H. Infanta Elena,0.230769
3,H. Infanta Margarita,23.801170
4,H. La Inmaculada,1.112485
5,H. La Merced,0.740056
6,H. Punta de Europa,0.374181
7,H. San Agustín,1.198801
8,H. San Juan de Dios del Aljarafe,94.225481
9,H. San Juan de la Cruz,7.324841


In [21]:
# Filtro A: Quitamos los centros de la "lista negra"
patient = patient[~patient['centro'].isin(centros_a_eliminar)]

# Filtro B: Nos quedamos solo con los pacientes que están en lab
# (Hacemos el .copy() aquí para que sea la tabla definitiva)
patient = patient[patient['NUHSA_ENCRIPTADO'].isin(lab['NUHSA_ENCRIPTADO'])].copy()

# 5. Resultado final
print(f"Pacientes finales tras limpiar centros y registros sin lab: {len(patient)}")

Pacientes finales tras limpiar centros y registros sin lab: 52464


## Lab data preprocess


### Change DESC_CLC for the 'lab_x' codes

In [22]:
lab_map = {
            "Creatinina": "lab_creatinine",                         # unidad mg/dl    (503)                  
            "Glucosa": "lab_glucose",                               # unidad mg/dl    (500)
            "Aspartato transaminasa": "lab_ast",                    # unidad U/L      (541)
            "Alanina transaminasa": "lab_alt",                      # unidad U/L      (542)
            "Potasio": "lab_potassium",                             # unidad mEq/L    (508), en HM unidad mmol/L                     
            "Sodio": "lab_sodium",                                  # unidad mEq/L    (506), en HM unidad mmol/L
            "Hematocrito": "lab_hct",                               # unidad %        (193)
            "Hemoglobina": "lab_hemoglobin",                        # unidad g/DL     (195)
            "Linfocitos (porcentaje)": "lab_lymphocyte_percentage", # unidad %        (175)
            "Linfocitos (recuento)": "lab_lymphocyte",              # unidad x10^3/µL (198) 
            "Hemoglobina corpuscular media": "lab_mch",             # unidad pg       (197)
            "Volumen corpuscular medio": "lab_mcv",                 # unidad fL       (194)
            "Neutrófilos (porcentaje)": "lab_neutrophil_percentage",# unidad %                     
            "Neutrófilos (recuento)": "lab_neutrophil",             # unidad x10^3/µL (174)
            "Plaquetas (recuento)": "lab_platelet",                 # unidad x10^3/µL (198)
            "Hematíes (recuento)": "lab_rbc",                       # unidad x10^6/µL (192)
            "Leucocitos (recuento)": "lab_leukocyte",               # unidad x10^3/µL (173)
            "Urea": "lab_urea",                                     # unidad mg/dl    (503)                         
            "Lactato deshidrogenasa": "lab_ldh",                    # unidad U/L      (525)
            "Proteína C reactiva": "lab_crp",                       # unidad mg/L     (636)   
            "Tiempo de protrombina normalizado (INR)": "lab_inr",   # unidad -        (164)
            "Dímero-D": "lab_ddimer",                               # unidad ng/mL    (115)                
            "Tiempo de protrombina": "lab_prothrombin",             # unidad NA       (266)  - equivalencia con HM desconocida 
}

In [23]:
# Convert non-numeric values to NaN
cols_medicas = ['VALOR_CONVENCIONAL', 'RANGO_INFERIOR_CONVENCIONAL', 'RANGO_SUPERIOR_CONVENCIONAL']

for col in cols_medicas:
    # Convertimos a string, cambiamos coma por punto y luego a numérico
    lab[col] = pd.to_numeric(lab[col].astype(str).str.replace(',', '.'), errors='coerce')
    
lab['DESC_CLC'] = lab['DESC_CLC'].str.strip() # remove whitespaces from the beginning and end of the strings

# Solo nos quedamos con lo que está en tu mapa (borramos el resto)
lab_filtered = lab[lab['DESC_CLC'].isin(lab_map.keys())].copy()

# Renombramos las pruebas para que tengan nombres estándar (ej: 'lab_glucose')
lab_filtered['DESC_CLC'] = lab_filtered['DESC_CLC'].map(lab_map)

In [24]:
lab_patient = pd.merge(
    patient[['NUHSA_ENCRIPTADO', 'admission_datetime',]], 
    lab_filtered, 
    on='NUHSA_ENCRIPTADO', 
    how='inner'
)

In [25]:
# 2. Reordenamos las columnas para que las fechas estén "al frente"
# Ponemos primero los IDs y las fechas, luego el resto de analíticas
columnas_fechas = ['NUHSA_ENCRIPTADO', 'admission_datetime', 'FEC_EXTRAE']
resto_columnas = [c for c in lab_patient.columns if c not in columnas_fechas]

lab_patient = lab_patient[columnas_fechas + resto_columnas].copy()

# 3. Ahora el cálculo de las 24h es súper sencillo:
lab_patient['horas_desde_ingreso'] = (lab_patient['FEC_EXTRAE'] - lab_patient['admission_datetime']).dt.total_seconds() / 3600

In [26]:
# =====================================================================
# 1. FILTRADO A 72H Y CREACIÓN DE BANDERAS (Formato Largo)
# =====================================================================
# 🌟 ¡EL CAMBIO CLAVE!: Filtramos antes de hacer nada más
df_72h_long = lab_patient[lab_patient['horas_desde_ingreso'] <= 72].copy()

# Aseguramos numéricos en los datos filtrados
cols_numericas = ['VALOR_CONVENCIONAL', 'RANGO_INFERIOR_CONVENCIONAL', 'RANGO_SUPERIOR_CONVENCIONAL']
for col in cols_numericas:
    df_72h_long[col] = pd.to_numeric(df_72h_long[col], errors='coerce')

# Calculamos las banderas SOLO para esas 72 horas
df_72h_long['below'] = (df_72h_long['VALOR_CONVENCIONAL'] < df_72h_long['RANGO_INFERIOR_CONVENCIONAL']).astype(float)
df_72h_long['over'] = (df_72h_long['VALOR_CONVENCIONAL'] > df_72h_long['RANGO_SUPERIOR_CONVENCIONAL']).astype(float)


# =====================================================================
# 2. PIVOTADO (Ahora súper rápido porque hay muchos menos datos)
# =====================================================================
df_wide = df_72h_long.pivot_table(
    index=['NUHSA_ENCRIPTADO', 'admission_datetime', 'FEC_EXTRAE', 'horas_desde_ingreso'], 
    columns='DESC_CLC',
    values=['VALOR_CONVENCIONAL', 'below', 'over'],
    aggfunc='first'
)

# Aplanar el multi-índice y limpiar nombres
nuevas_columnas = []
for medida, prueba in df_wide.columns:
    if medida == 'VALOR_CONVENCIONAL':
        nuevas_columnas.append(f"{prueba}_val")
    else:
        nuevas_columnas.append(f"{prueba}_{medida}")

df_wide.columns = nuevas_columnas
df_wide = df_wide.reset_index()

In [27]:
# =====================================================================
# 3. LA EXTRACCIÓN A 72 HORAS (Agrupación Final)
# =====================================================================
# Ordenamos cronológicamente
df_wide = df_wide.sort_values(by=['NUHSA_ENCRIPTADO', 'horas_desde_ingreso'])

# Crear el diccionario dinámico de agregación
agg_dict = {}
for col in df_wide.columns:
    if col.endswith('_val'):
        agg_dict[col] = ['first', 'last', 'min', 'max']
    elif col.endswith('_below') or col.endswith('_over'):
        agg_dict[col] = ['max']

# Agrupar por paciente (ahora condensa la peli de 3 días en 1 fila)
df_final_72h = df_wide.groupby('NUHSA_ENCRIPTADO').agg(agg_dict)

# Aplanar nombres de columnas
df_final_72h.columns = [f"{col}_{metrica}" for col, metrica in df_final_72h.columns]
df_final_72h = df_final_72h.reset_index()


# =====================================================================
# 4. CALCULAR LOS DELTAS Y RECUPERAR FECHA DE INGRESO
# =====================================================================
columnas_pruebas = [c.replace('_val_first', '') for c in df_final_72h.columns if c.endswith('_val_first')]

for prueba in columnas_pruebas:
    col_first = f"{prueba}_val_first"
    col_last = f"{prueba}_val_last"
    if col_first in df_final_72h.columns and col_last in df_final_72h.columns:
        df_final_72h[f"{prueba}_val_delta"] = df_final_72h[col_last] - df_final_72h[col_first]

# Recuperar la fecha de ingreso desde el df original filtrado
ingresos = df_72h_long[['NUHSA_ENCRIPTADO', 'admission_datetime']].drop_duplicates(subset=['NUHSA_ENCRIPTADO'])
df_final_72h = pd.merge(ingresos, df_final_72h, on='NUHSA_ENCRIPTADO', how='left')

In [28]:
patient_lab = pd.merge(
    patient, 
    df_final_72h.drop(columns=['admission_datetime'], errors='ignore'), 
    on='NUHSA_ENCRIPTADO', 
    how='inner' # <-- PARA NO VOLVER A METER PACIENTES SIN LAB
)

In [29]:
patient_lab.drop(columns=['completitud','FEC_EXTRAE'], errors='ignore', inplace=True)

In [30]:
patient_lab.columns

Index(['NUHSA_ENCRIPTADO', 'COD_FEC_NACIMIENTO', 'sex', 'centro', 'uci',
       'u_stay', 'es_estancia_unica', 'admission_datetime',
       'discharge_datetime', 'hospitalized_days',
       ...
       'lab_lymphocyte_percentage_val_delta', 'lab_mch_val_delta',
       'lab_mcv_val_delta', 'lab_neutrophil_val_delta',
       'lab_neutrophil_percentage_val_delta', 'lab_platelet_val_delta',
       'lab_potassium_val_delta', 'lab_rbc_val_delta', 'lab_sodium_val_delta',
       'lab_urea_val_delta'],
      dtype='object', length=169)

### Move to pmhx 

In [31]:
patologias_unicas = pmhx['DESC_PATOLOGIA_CRONICA'].unique()
print(patologias_unicas)

['Artrosis, espondilosis' 'Dislipemia' 'Hipotiroidismo' 'Hipertensión'
 'Demencia' 'Otro trastorno mental orgánico' 'Obesidad'
 'Arteriopatía de extremidades' 'Enfermedad valvular adquirida' 'Glaucoma'
 'Gota y otras artropatías por cristales' 'EPOC' 'Trastorno estado animo'
 'Retinopatía' 'Insuficiencia cardiaca' 'Insuficiencia renal crónica'
 'Trastorno de ansiedad' 'Diabetes' 'Cardiopatía isquémica'
 'Litiasis urinaria' 'Trastorno inicio infancia adolescencia'
 'Dependencia tabaco' 'Aneurisma arterias aorta, periféricas y viscerales'
 'Hepatopatía crónica excepto cirrosis' 'Dependencia alcohol'
 'Fibrilación auricular' 'Otra artropatía' 'Esteatosis hepática'
 'Secuela enfermedad cerebrovascular'
 'Enfermedad neurológica con déficit motor no ACV' 'Asma'
 'Degeneración macular asociada a la edad' 'Cáncer de mama'
 'Cáncer de próstata' 'Arteriopatía intraabdominal'
 'Trastorno conducta alimentaria' 'Osteoporosis' 'Enfermedad de Parkinson'
 'Trastorno esquizofrénico' 'Cáncer de cabeza y

In [32]:
dx_dict = {
    # --- LOS BIG 4 DEL COVID (Predictores top) ---
    'pmhx_obesity': ['Obesidad'], 
    'pmhx_diabetes': ['Diabetes', 'Retinopatía'],
    'pmhx_ckd': ['Insuficiencia renal crónica'],
    'pmhx_chf': ['Insuficiencia cardiaca'],

    # --- RESPIRATORIO (Directamente afectado por COVID) ---
    'pmhx_copd_asthma': ['EPOC', 'Asma'],

    # --- INMUNOSUPRESIÓN (Riesgo máximo de tormenta de citoquinas) ---
    'pmhx_immunosuppression': [
        'VIH', 'Sarcoma de Kaposi', 'Leucemia', 'Linfoma no Hodgkin', 
        'Enfermedad de Hodgkin', 'Cáncer inmunoproliferativo',
        'Enfermedad del colágeno y vasculitis', 
        'Artritis reumatoide y enfermedades relacionadas'
        # + El resto de cánceres de tu lista anterior...
    ],

    # --- CARDIOVASCULAR (El COVID causa miocarditis y trombos) ---
    'pmhx_afib': ['Fibrilación auricular'],
    'pmhx_ihd': ['Cardiopatía isquémica'],
    'pmhx_vascular': [
        'Arteriopatía de extremidades', 'Aneurisma arterias aorta, periféricas y viscerales',
        'Secuela enfermedad cerebrovascular', 'Enfermedad cerebrovascular mal definida y otra'
    ],

    # --- FRAGILIDAD (Marcadores de alta vulnerabilidad) ---
    'pmhx_frailty': [
        'Demencia', 'Otro trastorno mental orgánico', 
        'Enfermedad de Parkinson', 'Discapacidad intelectual'
    ],

    # --- OTROS ---
    'pmhx_liver': ['Cirrosis hepática', 'Hepatopatía crónica excepto cirrosis'],
    'pmhx_tobacco': ['Dependencia tabaco']
}

In [33]:
# 1. Empezamos con la lista de todos tus pacientes
pmhx_final = patient[['NUHSA_ENCRIPTADO']].copy()

# 2. Iteramos por el diccionario y creamos las columnas
for columna, lista_enfermedades in dx_dict.items():
    # Buscamos qué pacientes tienen alguna de esas enfermedades en pmhx
    mask = pmhx['DESC_PATOLOGIA_CRONICA'].isin(lista_enfermedades)
    pacientes_con_dx = pmhx.loc[mask, 'NUHSA_ENCRIPTADO'].unique()
    
    # Marcamos con 1 si el ID está en esa lista, y con 0 si no
    pmhx_final[columna] = pmhx_final['NUHSA_ENCRIPTADO'].isin(pacientes_con_dx).astype(int)

In [34]:
porcentajes = pmhx_final.drop(columns='NUHSA_ENCRIPTADO').mean() * 100

# 2. Lo convertimos a un DataFrame para que se vea más limpio
df_porcentajes = porcentajes.reset_index()
df_porcentajes.columns = ['Comorbilidad', 'Porcentaje (%)']

# 3. Ordenamos de mayor a menor frecuencia
df_porcentajes = df_porcentajes.sort_values(by='Porcentaje (%)', ascending=False)

# 4. Mostramos el resultado con 2 decimales
print(f"Análisis de prevalencia (N total = {len(pmhx_final)}):")
display(df_porcentajes.style.format({'Porcentaje (%)': '{:.2f}%'}))

Análisis de prevalencia (N total = 52464):


,Comorbilidad,Porcentaje (%)
1,pmhx_diabetes,38.30%
4,pmhx_copd_asthma,29.04%
0,pmhx_obesity,25.28%
3,pmhx_chf,22.87%
9,pmhx_frailty,19.02%
2,pmhx_ckd,16.83%
6,pmhx_afib,16.36%
8,pmhx_vascular,16.12%
7,pmhx_ihd,12.31%
11,pmhx_tobacco,12.27%


In [35]:
pmhx_clean = pmhx_final.drop_duplicates(subset=['NUHSA_ENCRIPTADO'])

In [36]:
print(f"Filas en patient_lab: {len(patient_lab)}")
print(f"IDs duplicados en patient_lab: {patient_lab['NUHSA_ENCRIPTADO'].duplicated().sum()}")
print(f"IDs duplicados en pmhx_final: {pmhx_clean['NUHSA_ENCRIPTADO'].duplicated().sum()}")

Filas en patient_lab: 51901
IDs duplicados en patient_lab: 1225
IDs duplicados en pmhx_final: 0


In [37]:
patient_complete = pd.merge(patient_lab, pmhx_clean, on='NUHSA_ENCRIPTADO', how='left')

# Las columnas de comorbilidades que acabamos de añadir tendrán NaNs 
# en los pacientes que no tenían registros en pmhx. Los llenamos con 0.
cols_pmhx = [c for c in pmhx_clean.columns if c.startswith('pmhx_')]
patient_complete[cols_pmhx] = patient_complete[cols_pmhx].fillna(0).astype(int)
patient_complete.columns

Index(['NUHSA_ENCRIPTADO', 'COD_FEC_NACIMIENTO', 'sex', 'centro', 'uci',
       'u_stay', 'es_estancia_unica', 'admission_datetime',
       'discharge_datetime', 'hospitalized_days',
       ...
       'pmhx_ckd', 'pmhx_chf', 'pmhx_copd_asthma', 'pmhx_immunosuppression',
       'pmhx_afib', 'pmhx_ihd', 'pmhx_vascular', 'pmhx_frailty', 'pmhx_liver',
       'pmhx_tobacco'],
      dtype='object', length=181)

In [38]:
periodos
periodos_data = [
    (1, '2020-01-01', '2020-05-10'),
    (2, '2020-05-11', '2020-12-20'),
    (3, '2020-12-21', '2021-03-07'),
    (4, '2021-03-08', '2021-06-20'),
    (5, '2021-06-21', '2021-10-10'),
    (6, '2021-10-11', '2022-03-27'),
    (7, '2022-03-28', '2022-11-01')
]

In [39]:
# 3. Función para asignar el periodo
def asignar_periodo(fecha):
    if pd.isna(fecha):
        return np.nan
    for num, inicio, fin in periodos_data:
        if pd.to_datetime(inicio) <= fecha <= pd.to_datetime(fin):
            return num
    return np.nan # Por si alguna fecha cae fuera de los rangos

# 4. Aplicamos la lógica
patient_complete['covid_period'] = patient_complete['admission_datetime'].apply(asignar_periodo)

### Vacunas

In [40]:
# 1. Obtenemos los IDs únicos de ambas tablas
ids_vacunas = set(vacunas['NUHSA_ENCRIPTADO'].unique())
ids_patient = set(patient_complete['NUHSA_ENCRIPTADO'].unique())

# 2. Calculamos la diferencia (lo que está en vacunas pero NO en patient)
huerfanos_vacunas = ids_vacunas - ids_patient

# 3. Resultados
print(f"📊 Pacientes únicos en tabla 'vacunas': {len(ids_vacunas)}")
print(f"📊 Pacientes únicos en tabla 'patient': {len(ids_patient)}")
print(f"⚠️ Pacientes en 'vacunas' que NO existen en 'patient': {len(huerfanos_vacunas)}")

📊 Pacientes únicos en tabla 'vacunas': 45707
📊 Pacientes únicos en tabla 'patient': 50676
⚠️ Pacientes en 'vacunas' que NO existen en 'patient': 4999


In [41]:
vacunas_clean = vacunas[vacunas['NUHSA_ENCRIPTADO'].isin(patient['NUHSA_ENCRIPTADO'])].copy()

In [42]:
print(vacunas_clean['IND_GRUPO_RIESGO'].value_counts())

IND_GRUPO_RIESGO
1    113867
0      4925
Name: count, dtype: int64


In [43]:
def get_vacunas(patient, vacunas):
    # 1. Aseguramos que las fechas sean datetime (vital para el cálculo)
    patient['admission_datetime'] = pd.to_datetime(patient['admission_datetime'])
    vacunas['FEC_VACUNACION'] = pd.to_datetime(vacunas['FEC_VACUNACION'])

    # 2. Hacemos un "merge" de todas las vacunas con todos los ingresos
    # Esto crea una fila por cada combinación de (Ingreso, Vacuna) de un mismo paciente
    df_merge = pd.merge(
        patient[['NUHSA_ENCRIPTADO', 'admission_datetime']], 
        vacunas[['NUHSA_ENCRIPTADO', 'FEC_VACUNACION']], 
        on='NUHSA_ENCRIPTADO', 
        how='inner'
    )

    # 3. Aplicamos tu regla: Fecha Vacuna + 14 días < Fecha Ingreso
    # Creamos una máscara booleana para filtrar
    condicion_efectiva = (df_merge['FEC_VACUNACION'] + pd.Timedelta(days=14)) < df_merge['admission_datetime']
    df_merge = df_merge[condicion_efectiva]

    # 4. Contamos cuántas vacunas han quedado para cada ingreso de cada paciente
    shots_count = df_merge.groupby(['NUHSA_ENCRIPTADO', 'admission_datetime']).size().reset_index(name='num_shots')

    # 5. Lo pegamos a la tabla original de pacientes
    # Usamos left join para que los que no tienen vacunas se queden con NaN
    patient = pd.merge(patient, shots_count, on=['NUHSA_ENCRIPTADO', 'admission_datetime'], how='left')

    # 6. Limpieza final: rellenar los que no tienen ninguna dosis con 0 y pasar a entero
    patient['num_shots'] = patient['num_shots'].fillna(0).astype(int)

    return patient

In [44]:
patient_vacunas = get_vacunas(patient_complete,vacunas_clean)

In [45]:
patient_vacunas['is_vaccinated'] = (patient_vacunas['num_shots'] > 0).astype(int)

In [46]:
patient_vacunas.columns

Index(['NUHSA_ENCRIPTADO', 'COD_FEC_NACIMIENTO', 'sex', 'centro', 'uci',
       'u_stay', 'es_estancia_unica', 'admission_datetime',
       'discharge_datetime', 'hospitalized_days',
       ...
       'pmhx_immunosuppression', 'pmhx_afib', 'pmhx_ihd', 'pmhx_vascular',
       'pmhx_frailty', 'pmhx_liver', 'pmhx_tobacco', 'covid_period',
       'num_shots', 'is_vaccinated'],
      dtype='object', length=184)

In [47]:
cols_to_remove = ['NUHSA_ENCRIPTADO',
                  'discharge_datetime',
                  'COD_FEC_FALLECIMIENTO', 
                  'death_datetime',
                 # 'pmhx_tobacco',
                  'COD_FEC_NACIMIENTO',
                  'admission_datetime',
                  'hospitalized_days',
                  'uci',
                  'es_estancia_unica',
                  'u_stay',
                    ]
cols_to_remove = [col for col in cols_to_remove if col in patient_vacunas.columns]
patient_vacunas.drop(columns=cols_to_remove, inplace=True)
patient_vacunas.columns

Index(['sex', 'centro', 'hospital_outcome', 'age', 'lab_alt_val_first',
       'lab_alt_val_last', 'lab_alt_val_min', 'lab_alt_val_max',
       'lab_ast_val_first', 'lab_ast_val_last',
       ...
       'pmhx_immunosuppression', 'pmhx_afib', 'pmhx_ihd', 'pmhx_vascular',
       'pmhx_frailty', 'pmhx_liver', 'pmhx_tobacco', 'covid_period',
       'num_shots', 'is_vaccinated'],
      dtype='object', length=175)

In [48]:
patient_vacunas = pd.get_dummies(
    patient_vacunas, 
    columns=['covid_period'], 
    prefix='periodo',      # Crea nombres limpios como 'periodo_1.0', 'periodo_2.0'
    dummy_na=False,        # Cambia a True si quieres una columna específica para los NaN
    dtype=int              # CRÍTICO: Fuerza a que sean 0 y 1, no booleanos True/False
)

In [50]:
patient_vacunas.to_pickle(pickle_path+'/andalucia_macro.pickle')

In [51]:
patient_vacunas

,sex,centro,hospital_outcome,age,lab_alt_val_first,lab_alt_val_last,lab_alt_val_min,lab_alt_val_max,lab_ast_val_first,lab_ast_val_last,lab_ast_val_min,lab_ast_val_max,lab_creatinine_val_first,lab_creatinine_val_last,lab_creatinine_val_min,lab_creatinine_val_max,lab_crp_val_first,lab_crp_val_last,lab_crp_val_min,lab_crp_val_max,lab_ddimer_val_first,lab_ddimer_val_last,lab_ddimer_val_min,lab_ddimer_val_max,lab_glucose_val_first,lab_glucose_val_last,lab_glucose_val_min,lab_glucose_val_max,lab_hct_val_first,lab_hct_val_last,lab_hct_val_min,lab_hct_val_max,lab_hemoglobin_val_first,lab_hemoglobin_val_last,lab_hemoglobin_val_min,lab_hemoglobin_val_max,lab_inr_val_first,lab_inr_val_last,lab_inr_val_min,lab_inr_val_max,lab_ldh_val_first,lab_ldh_val_last,lab_ldh_val_min,lab_ldh_val_max,lab_leukocyte_val_first,lab_leukocyte_val_last,lab_leukocyte_val_min,lab_leukocyte_val_max,lab_lymphocyte_val_first,lab_lymphocyte_val_last,lab_lymphocyte_val_min,lab_lymphocyte_val_max,lab_lymphocyte_percentage_val_first,lab_lymphocyte_percentage_val_last,lab_lymphocyte_percentage_val_min,lab_lymphocyte_percentage_val_max,lab_mch_val_first,lab_mch_val_last,lab_mch_val_min,lab_mch_val_max,lab_mcv_val_first,lab_mcv_val_last,lab_mcv_val_min,lab_mcv_val_max,lab_neutrophil_val_first,lab_neutrophil_val_last,lab_neutrophil_val_min,lab_neutrophil_val_max,lab_neutrophil_percentage_val_first,lab_neutrophil_percentage_val_last,lab_neutrophil_percentage_val_min,lab_neutrophil_percentage_val_max,lab_platelet_val_first,lab_platelet_val_last,lab_platelet_val_min,lab_platelet_val_max,lab_potassium_val_first,lab_potassium_val_last,lab_potassium_val_min,lab_potassium_val_max,lab_rbc_val_first,lab_rbc_val_last,lab_rbc_val_min,lab_rbc_val_max,lab_sodium_val_first,lab_sodium_val_last,lab_sodium_val_min,lab_sodium_val_max,lab_urea_val_first,lab_urea_val_last,lab_urea_val_min,lab_urea_val_max,lab_alt_below_max,lab_ast_below_max,lab_creatinine_below_max,lab_crp_below_max,lab_ddimer_below_max,lab_glucose_below_max,lab_hct_below_max,lab_hemoglobin_below_max,lab_inr_below_max,lab_ldh_below_max,lab_leukocyte_below_max,lab_lymphocyte_below_max,lab_lymphocyte_percentage_below_max,lab_mch_below_max,lab_mcv_below_max,lab_neutrophil_below_max,lab_neutrophil_percentage_below_max,lab_platelet_below_max,lab_potassium_below_max,lab_prothrombin_below_max,lab_rbc_below_max,lab_sodium_below_max,lab_urea_below_max,lab_alt_over_max,lab_ast_over_max,lab_creatinine_over_max,lab_crp_over_max,lab_ddimer_over_max,lab_glucose_over_max,lab_hct_over_max,lab_hemoglobin_over_max,lab_inr_over_max,lab_ldh_over_max,lab_leukocyte_over_max,lab_lymphocyte_over_max,lab_lymphocyte_percentage_over_max,lab_mch_over_max,lab_mcv_over_max,lab_neutrophil_over_max,lab_neutrophil_percentage_over_max,lab_platelet_over_max,lab_potassium_over_max,lab_prothrombin_over_max,lab_rbc_over_max,lab_sodium_over_max,lab_urea_over_max,lab_alt_val_delta,lab_ast_val_delta,lab_creatinine_val_delta,lab_crp_val_delta,lab_ddimer_val_delta,lab_glucose_val_delta,lab_hct_val_delta,lab_hemoglobin_val_delta,lab_inr_val_delta,lab_ldh_val_delta,lab_leukocyte_val_delta,lab_lymphocyte_val_delta,lab_lymphocyte_percentage_val_delta,lab_mch_val_delta,lab_mcv_val_delta,lab_neutrophil_val_delta,lab_neutrophil_percentage_val_delta,lab_platelet_val_delta,lab_potassium_val_delta,lab_rbc_val_delta,lab_sodium_val_delta,lab_urea_val_delta,pmhx_obesity,pmhx_diabetes,pmhx_ckd,pmhx_chf,pmhx_copd_asthma,pmhx_immunosuppression,pmhx_afib,pmhx_ihd,pmhx_vascular,pmhx_frailty,pmhx_liver,pmhx_tobacco,num_shots,is_vaccinated,periodo_1.0,periodo_2.0,periodo_3.0,periodo_4.0,periodo_5.0,periodo_6.0,periodo_7.0
0,0,H. de Baza,0,26,237.0,237.0,237.0,237.0,NaN,NaN,NaN,NaN,0.83,0.85,0.76,0.86,76.3,9.6,9.6,76.3,950.0,950.0,950.0,950.0,145.0,116.0,116.0,145.0,42.2,38.8,38.8,42.7,14.8,12.9,12.9,14.8,1.03,1.17,1.02,1.28,856.0,856.0,856.0,856.0,4.75,5.70,4.41,5.70,1.08,0.79,0.74,1.08,22.70,13.90,13.90,22.70,29.8,29.2,28.7,29.8,84.9,87.8,84.9,88.3,3.29,4.43,3.29,4.43,69.20,77.70,69.